# Notebook 01: Fine-Tuning BGE-Small with Contrastive Learning
## Hospital RAG Model Training and Comparative Research

### 1. Research Objectives
1. **Base Model**: Fine-tune `BAAI/bge-small-en-v1.5` (~33M parameters, 384 dimensions).
2. **Domain Adaptation**: Adapt the general embedding space to hospital and medical query-passage relationships using MedQuAD.
3. **Contrastive Loss**: Use **MultipleNegativesRankingLoss** (MNRL / InfoNCE), which maximizes the similarity between patient questions and verified clinical answers while treating all other answers in the batch as negative distractors.
4. **Dimension Preservation**: Ensure the final model retains its **384-dimensional** embedding output, guaranteeing 100% plug-and-play compatibility with your project's Supabase database schema (`vector(384)`).
5. **Hardware Safety**: Optimized for Windows with an NVIDIA RTX 3050 (6GB VRAM) and 8GB RAM, utilizing mixed precision (`fp16`) and conservative batch sizing (8).

---
### 2. Training Strategy
- **Mode Toggle**: Includes `USE_QUICK_TEST_MODE = True` (runs on 300 samples in ~2 minutes to verify end-to-end functionality before full training).
- **Validation**: Tracks validation loss and retrieval performance after every epoch.
- **Checkpointing**: Saves the best model weights to `experiments/models/finetuned_bge_small/`.

In [21]:
# Cell 1: Environment & Hardware Detection
import os
import sys
import gc
import platform
import psutil
import torch

print("=" * 60)
print("COMPUTE ENVIRONMENT & MEMORY DIAGNOSTICS")
print("=" * 60)

# Memory cleanup before initialization
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device_name = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target Compute Device : {device_name.upper()}")

if device_name == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_free = torch.cuda.mem_get_info()[0]
    print(f"GPU Model             : {gpu_name}")
    print(f"Total VRAM            : {vram_bytes / (1024**3):.2f} GB")
    print(f"Available Free VRAM   : {vram_free / (1024**3):.2f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    print("WARNING: CUDA is not available. Falling back to multi-core CPU training.")

ram = psutil.virtual_memory()
print(f"System RAM Available  : {ram.available / (1024**3):.2f} GB ({ram.percent}% used)")
print("=" * 60)

COMPUTE ENVIRONMENT & MEMORY DIAGNOSTICS
Target Compute Device : CPU
System RAM Available  : 0.64 GB (91.6% used)


In [22]:
# Cell 2: Imports & Experiment Configuration
from pathlib import Path
import json
import time
import pandas as pd
import numpy as np

# Set project root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

CONFIG_PATH = PROJECT_ROOT / "experiments" / "config" / "experiment_config.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

PROCESSED_DATA_DIR = PROJECT_ROOT / config["paths"]["processed_data_dir"]
MODEL_OUTPUT_DIR = PROJECT_ROOT / config["paths"]["finetuned_model_path"]
RESULTS_DIR = PROJECT_ROOT / config["paths"]["results_dir"] / "finetuning"

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root     : {PROJECT_ROOT}")
print(f"Model Output Dir : {MODEL_OUTPUT_DIR}")
print(f"Results Dir      : {RESULTS_DIR}")

Project Root     : d:\AI Hospital Assistant
Model Output Dir : d:\AI Hospital Assistant\experiments\models\finetuned_bge_small
Results Dir      : d:\AI Hospital Assistant\experiments\results\finetuning


In [23]:
# Cell 3: Training Hyperparameters & Mode Selection
# ==============================================================================
# USER CONFIGURATION: Set USE_QUICK_TEST_MODE to True for a fast verification run!
# ==============================================================================
USE_QUICK_TEST_MODE = True    # Set to False when you are ready for full training

if USE_QUICK_TEST_MODE:
    print(">>> RUNNING IN QUICK TEST MODE (Fast verification on subset) <<<")
    BATCH_SIZE = config["hardware"]["quick_test_batch_size"]  # 4
    EPOCHS = 1
    TRAIN_SAMPLE_LIMIT = config["dataset"]["quick_test_subset_size"]  # 300
    VAL_SAMPLE_LIMIT = 50
    EVAL_STEPS = 25
else:
    print(">>> RUNNING IN FULL TRAINING MODE <<<")
    BATCH_SIZE = config["hardware"]["default_train_batch_size"]  # 8 (Safe for 6GB VRAM)
    EPOCHS = config["training"]["epochs"]                        # 3
    TRAIN_SAMPLE_LIMIT = config["dataset"]["full_train_subset_size"] # 10,000
    VAL_SAMPLE_LIMIT = 1000
    EVAL_STEPS = config["training"]["evaluation_steps"]          # 250

LEARNING_RATE = float(config["training"]["learning_rate"])       # 2e-5
BASE_MODEL_NAME = config["training"]["base_model"]               # BAAI/bge-small-en-v1.5
MAX_SEQ_LENGTH = config["training"]["max_seq_length"]            # 512
USE_FP16 = config["hardware"]["use_fp16"] and (device_name == "cuda")

print(f"Base Model    : {BASE_MODEL_NAME}")
print(f"Batch Size    : {BATCH_SIZE}")
print(f"Epochs        : {EPOCHS}")
print(f"Learning Rate : {LEARNING_RATE}")
print(f"Mixed Prec.   : fp16 = {USE_FP16}")
print(f"Max Seq Len   : {MAX_SEQ_LENGTH}")

>>> RUNNING IN QUICK TEST MODE (Fast verification on subset) <<<
Base Model    : BAAI/bge-small-en-v1.5
Batch Size    : 4
Epochs        : 1
Learning Rate : 2e-05
Mixed Prec.   : fp16 = False
Max Seq Len   : 512


In [24]:
# Cell 4: Load Prepared Datasets
train_csv = PROCESSED_DATA_DIR / "train.csv"
val_csv = PROCESSED_DATA_DIR / "validation.csv"

assert train_csv.exists(), f"Train dataset missing at {train_csv}. Run Notebook 00 first!"
assert val_csv.exists(), f"Validation dataset missing at {val_csv}. Run Notebook 00 first!"

df_train = pd.read_csv(train_csv)
df_val = pd.read_csv(val_csv)

if TRAIN_SAMPLE_LIMIT and len(df_train) > TRAIN_SAMPLE_LIMIT:
    df_train = df_train.sample(n=TRAIN_SAMPLE_LIMIT, random_state=42).reset_index(drop=True)

if VAL_SAMPLE_LIMIT and len(df_val) > VAL_SAMPLE_LIMIT:
    df_val = df_val.sample(n=VAL_SAMPLE_LIMIT, random_state=42).reset_index(drop=True)

print(f"Loaded Training Samples   : {len(df_train):,}")
print(f"Loaded Validation Samples : {len(df_val):,}")
display(df_train[["question", "answer", "focus"]].head(2))

Loaded Training Samples   : 300
Loaded Validation Samples : 50


,question,answer,focus
0,Is mitochondrial complex III deficiency inheri...,Mitochondrial complex III deficiency is usuall...,inheritance
1,What are the symptoms of BRCA1 hereditary brea...,What are the signs and symptoms of BRCA1 hered...,symptoms


In [25]:
# Cell 5: Load Base Pretrained Model & Verify Embedding Dimension
from sentence_transformers import SentenceTransformer

print(f"Loading pretrained base model: {BASE_MODEL_NAME} ...")
model = SentenceTransformer(BASE_MODEL_NAME, device=device_name)
model.max_seq_length = MAX_SEQ_LENGTH

# Verify embedding dimension
test_vec = model.encode("Medical test query", normalize_embeddings=True)
embed_dim = test_vec.shape[0]

print("=" * 60)
print("BASE MODEL ARCHITECTURE VERIFICATION")
print("=" * 60)
print(f"Model ID            : {BASE_MODEL_NAME}")
print(f"Embedding Dimension : {embed_dim}")
print(f"Max Sequence Length : {model.max_seq_length}")
print(f"Target Supabase Dim : {config['training']['target_embedding_dimension']}")

assert embed_dim == config["training"]["target_embedding_dimension"], (
    f"Dimension mismatch! Model has {embed_dim}, but database expects "
    f"{config['training']['target_embedding_dimension']}"
)
print(">>> ARCHITECTURE CONFIRMED: 100% compatible with Supabase vector(384) schema. <<<")
print("=" * 60)

Loading pretrained base model: BAAI/bge-small-en-v1.5 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1696.02it/s]


BASE MODEL ARCHITECTURE VERIFICATION
Model ID            : BAAI/bge-small-en-v1.5
Embedding Dimension : 384
Max Sequence Length : 512
Target Supabase Dim : 384
>>> ARCHITECTURE CONFIRMED: 100% compatible with Supabase vector(384) schema. <<<


In [26]:
# Cell 6: Prepare Dataset Pairs and Loss Function
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

# Build query-positive passage training pairs
# Note: For BGE, prepending the recommended query instruction optimizes retrieval
bge_instruction = config["models_to_compare"][0].get("query_prefix", "")

train_examples = []
for _, row in df_train.iterrows():
    query_text = f"{bge_instruction}{row['question']}".strip()
    passage_text = str(row['answer']).strip()
    train_examples.append(InputExample(texts=[query_text, passage_text]))

train_dataloader = DataLoader(
    train_examples, 
    shuffle=True, 
    batch_size=BATCH_SIZE,
    collate_fn=model.smart_batching_collate,
    drop_last=True
)

# Configure MultipleNegativesRankingLoss
# For each pair (q_i, p_i) in the batch, p_j (where j != i) act as negative distractors
train_loss = losses.MultipleNegativesRankingLoss(model)

print(f"Created {len(train_examples):,} training pairs.")
print(f"Total Batches per Epoch: {len(train_dataloader):,}")
print(f"Loss Function: {train_loss.__class__.__name__} (In-Batch Contrastive Negatives)")

Created 300 training pairs.
Total Batches per Epoch: 75
Loss Function: MultipleNegativesRankingLoss (In-Batch Contrastive Negatives)


In [27]:
# Cell 7: Setup Validation Evaluator
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# Prepare validation corpus and queries for IR evaluation
val_queries = {}
val_corpus = {}
val_relevant_docs = {}

for idx, row in df_val.iterrows():
    qid = f"q_{idx}"
    cid = f"c_{idx}"
    val_queries[qid] = f"{bge_instruction}{row['question']}".strip()
    val_corpus[cid] = str(row['answer']).strip()
    val_relevant_docs[qid] = {cid}

evaluator = InformationRetrievalEvaluator(
    queries=val_queries,
    corpus=val_corpus,
    relevant_docs=val_relevant_docs,
    name="medquad-val",
    mrr_at_k=[5, 10],
    ndcg_at_k=[5, 10],
    accuracy_at_k=[1, 3, 5],
    precision_recall_at_k=[5],
    show_progress_bar=False
)

print(f"Configured InformationRetrievalEvaluator on {len(val_queries)} validation queries.")

Configured InformationRetrievalEvaluator on 50 validation queries.


In [28]:
# Cell 8: Pure PyTorch Contrastive Fine-Tuning Execution
# Completely bypasses HuggingFace Trainer and external datasets library for 100% Windows reliability
import math
import time
import torch
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

total_steps = len(train_dataloader) * EPOCHS
warmup_steps = math.ceil(total_steps * config["training"]["warmup_ratio"])

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16) if (device_name == "cuda" and USE_FP16) else None

print("=" * 60)
print("COMMENCING CONTRASTIVE FINE-TUNING (NATIVE PYTORCH)")
print("=" * 60)
print(f"Total Training Batches : {len(train_dataloader):,}")
print(f"Total Training Steps   : {total_steps:,}")
print(f"Warmup Steps           : {warmup_steps}")
print(f"Evaluation Steps       : {EVAL_STEPS}")
print(f"Mixed Precision (fp16) : {USE_FP16}")
print(f"Model Checkpoint Dir   : {MODEL_OUTPUT_DIR}")
print("=" * 60)

start_time = time.time()
best_val_score = -1.0
global_step = 0
training_history = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    epoch_start = time.time()
    
    for batch_idx, (features, labels) in enumerate(train_dataloader):
        optimizer.zero_grad()
        
        # Safely move feature tensors to target device (skipping non-tensor metadata like 'modality')
        features = [{k: (v.to(device_name) if hasattr(v, "to") else v) for k, v in f.items()} for f in features]
        if labels is not None and hasattr(labels, "to"):
            labels = labels.to(device_name)
            
        if USE_FP16 and scaler is not None:
            with torch.amp.autocast(device_type="cuda", enabled=True):
                loss = train_loss(features, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = train_loss(features, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
        scheduler.step()
        global_step += 1
        loss_val = loss.item()
        running_loss += loss_val
        
        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(train_dataloader):
            avg_step_loss = running_loss / (batch_idx + 1)
            lr_current = scheduler.get_last_lr()[0]
            print(f"Epoch [{epoch+1}/{EPOCHS}] | Step [{batch_idx+1}/{len(train_dataloader)}] | Loss: {loss_val:.4f} (Avg: {avg_step_loss:.4f}) | LR: {lr_current:.2e}")
            
        # Periodic evaluation
        if EVAL_STEPS > 0 and global_step % EVAL_STEPS == 0:
            eval_scores = evaluator(model)
            val_metric = eval_scores.get("medquad-val_cosine_ndcg@10", list(eval_scores.values())[0])
            print(f"  --> [Validation @ Step {global_step}] NDCG@10: {val_metric:.4f}")
            if val_metric > best_val_score:
                best_val_score = val_metric
                model.save_pretrained(str(MODEL_OUTPUT_DIR))
                print(f"  --> Saved new best checkpoint to {MODEL_OUTPUT_DIR}")
            model.train()

    epoch_dur = time.time() - epoch_start
    avg_epoch_loss = running_loss / max(1, len(train_dataloader))
    print(f"\n--- Epoch {epoch+1} Complete | Avg Loss: {avg_epoch_loss:.4f} | Time: {epoch_dur:.1f}s ---")
    
    # End-of-epoch evaluation
    eval_scores = evaluator(model)
    val_metric = eval_scores.get("medquad-val_cosine_ndcg@10", list(eval_scores.values())[0])
    print(f"--- Epoch {epoch+1} Validation NDCG@10: {val_metric:.4f} ---\n")
    if val_metric > best_val_score:
        best_val_score = val_metric
        model.save_pretrained(str(MODEL_OUTPUT_DIR))
        print(f"--- Saved new best model checkpoint! ---\n")
    
    training_history.append({
        "epoch": epoch + 1,
        "loss": avg_epoch_loss,
        "val_ndcg@10": val_metric
    })

training_duration_sec = time.time() - start_time
print("=" * 60)
print(f"FINE-TUNING COMPLETED in {training_duration_sec / 60:.2f} minutes ({training_duration_sec:.1f}s)")
print(f"Best Validation NDCG@10: {best_val_score:.4f}")
print("=" * 60)

COMMENCING CONTRASTIVE FINE-TUNING (NATIVE PYTORCH)
Total Training Batches : 75
Total Training Steps   : 75
Warmup Steps           : 8
Evaluation Steps       : 25
Mixed Precision (fp16) : False
Model Checkpoint Dir   : d:\AI Hospital Assistant\experiments\models\finetuned_bge_small
Epoch [1/1] | Step [10/75] | Loss: 0.0116 (Avg: 0.0474) | LR: 1.94e-05
Epoch [1/1] | Step [20/75] | Loss: 0.0160 (Avg: 0.0685) | LR: 1.64e-05
  --> [Validation @ Step 25] NDCG@10: 0.9926


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 12.65it/s]

  --> Saved new best checkpoint to d:\AI Hospital Assistant\experiments\models\finetuned_bge_small


Epoch [1/1] | Step [30/75] | Loss: 0.2605 (Avg: 0.0610) | LR: 1.34e-05
Epoch [1/1] | Step [40/75] | Loss: 0.1766 (Avg: 0.0531) | LR: 1.04e-05
Epoch [1/1] | Step [50/75] | Loss: 0.0087 (Avg: 0.0463) | LR: 7.46e-06
  --> [Validation @ Step 50] NDCG@10: 0.9926
Epoch [1/1] | Step [60/75] | Loss: 0.0017 (Avg: 0.0436) | LR: 4.48e-06
Epoch [1/1] | Step [70/75] | Loss: 0.0015 (Avg: 0.0389) | LR: 1.49e-06
Epoch [1/1] | Step [75/75] | Loss: 0.0015 (Avg: 0.0366) | LR: 0.00e+00
  --> [Validation @ Step 75] NDCG@10: 0.9926

--- Epoch 1 Complete | Avg Loss: 0.0366 | Time: 287.4s ---
--- Epoch 1 Validation NDCG@10: 0.9926 ---

FINE-TUNING COMPLETED in 4.85 minutes (290.9s)
Best Validation NDCG@10: 0.9926


In [29]:
# Cell 9: Save Final Model Artifacts and Training History
# Save the final fine-tuned model
model.save_pretrained(str(MODEL_OUTPUT_DIR))
print(f"Fine-tuned model weights saved to: {MODEL_OUTPUT_DIR}")

# Build training configuration log
training_summary = {
    "base_model": BASE_MODEL_NAME,
    "saved_model_path": str(MODEL_OUTPUT_DIR),
    "embedding_dimension": 384,
    "quick_test_mode": USE_QUICK_TEST_MODE,
    "train_samples_count": len(train_examples),
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "training_duration_seconds": round(training_duration_sec, 2),
    "best_val_ndcg@10": round(float(best_val_score), 4),
    "device_used": device_name,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
}

summary_json_path = RESULTS_DIR / "training_config.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(training_summary, f, indent=2)

print("Saved training configuration log to:", summary_json_path)

print("Saved training configuration log to:", summary_json_path)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.87it/s]

Fine-tuned model weights saved to: d:\AI Hospital Assistant\experiments\models\finetuned_bge_small
Saved training configuration log to: d:\AI Hospital Assistant\experiments\results\finetuning\training_config.json
Saved training configuration log to: d:\AI Hospital Assistant\experiments\results\finetuning\training_config.json


In [30]:
# Cell 10: Sanity Verification of Saved Fine-Tuned Model
print("Reloading saved fine-tuned model from disk for verification...")
reloaded_model = SentenceTransformer(str(MODEL_OUTPUT_DIR), device=device_name)

test_query = f"{bge_instruction}What are the complications of acute appendicitis?"
test_pos = "Complications of acute appendicitis include perforation, peritonitis, abscess formation, and pylephlebitis."
test_neg = "The parking garage is located adjacent to the west clinic tower."

v_q = reloaded_model.encode(test_query, normalize_embeddings=True)
v_pos = reloaded_model.encode(test_pos, normalize_embeddings=True)
v_neg = reloaded_model.encode(test_neg, normalize_embeddings=True)

sim_pos = float(np.dot(v_q, v_pos))
sim_neg = float(np.dot(v_q, v_neg))

print("=" * 60)
print("MODEL SANITY CHECK VERIFICATION")
print("=" * 60)
print(f"Query                     : {test_query}")
print(f"Embedding Shape           : {v_q.shape} (Expected: (384,))")
print(f"Similarity (Positive Med) : {sim_pos:.4f}")
print(f"Similarity (Negative Irrel): {sim_neg:.4f}")
print("=" * 60)

assert v_q.shape[0] == 384, f"Output dimension is {v_q.shape[0]}, expected 384!"
assert sim_pos > sim_neg, "Model failed semantic discrimination test!"

print(">>> VERIFICATION PASSED: The saved fine-tuned model is fully operational! <<<")

Reloading saved fine-tuned model from disk for verification...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3976.23it/s]

MODEL SANITY CHECK VERIFICATION
Query                     : Represent this sentence for searching relevant passages: What are the complications of acute appendicitis?
Embedding Shape           : (384,) (Expected: (384,))
Similarity (Positive Med) : 0.9289
Similarity (Negative Irrel): 0.2246
>>> VERIFICATION PASSED: The saved fine-tuned model is fully operational! <<<


### 3. Next Steps
Your fine-tuned embedding model is saved at **`experiments/models/finetuned_bge_small/`**.

**Next**: Open and run **`02_full_model_and_rag_comparison.ipynb`** to benchmark:
- Original `BAAI/bge-small-en-v1.5`
- `intfloat/e5-small-v2`
- `thenlper/gte-small`
- `all-MiniLM-L6-v2`
- **Your newly fine-tuned BGE-Small model!**